# Part 2 — Embeddings

> Part 2 of **[Tokens to Agents](../../README.md)** — the foundations of agentic AI, built from scratch.

Part 1 ended on a wall: an n-gram model trained on 4,846 financial headlines cannot tell that
*profit rose* and *earnings increased* say the same thing — every context is an island.
This notebook breaks that wall **on the same corpus**, twice:

1. **Counting** — co-occurrence → PPMI → SVD. No training loop at all.
2. **Predicting** — skip-gram with negative sampling, every gradient written by hand.

Then both are scored on the same test, and the result is not the one the last decade of
deep learning would have you guess.

Implementations follow [Jurafsky & Martin, *SLP* ch. 5](https://web.stanford.edu/~jurafsky/slp3/5.pdf).

In [ ]:
import numpy as np
from embeddings import (
    WordVectors,
    build_vocab,
    cooccurrence,
    encode,
    pair_auc,
    ppmi,
    random_pair_mean,
    svd_vectors,
    train_sgns,
)
from run import COUNT_CFG, PROMISE, RELATED, SGNS_CFG, SHOWCASE

from core import viz
from core.corpus import sentences

viz.use_theme()
sents = sentences()
vocab = build_vocab(sents, min_count=5)
ids = encode(sents, vocab)
print(f'{len(sents):,} sentences · {sum(len(s) for s in ids):,} in-vocab tokens · vocab {vocab.size:,}')

## Path 1 — count, then compress

Slide a ±2-token window over the corpus and count which words appear near which. PPMI keeps
only the co-occurrences that happen *more than chance predicts*; SVD compresses that matrix
into 100 dense dimensions. Words with similar rows land near each other — similarity by
construction, not by training.

In [ ]:
counted = WordVectors(vocab, svd_vectors(
    ppmi(cooccurrence(ids, vocab.size, window=COUNT_CFG['window'])), dim=COUNT_CFG['dim']))

for w in ('profit', 'rose', 'fell'):
    print(f'{w:<7}', ', '.join(f'{n} {s:.2f}' for n, s in counted.neighbors(w, 5)))

*profit*'s neighbours are its P&L family; *rose*'s are the other movement verbs —
including its own antonym. Co-occurrence captures **substitutability, not polarity**:
*rose* and *fell* appear in identical contexts, so they land side by side. Projected to 2-D:

In [ ]:
words = [w for ws in SHOWCASE.values() for w in ws]
groups = [g for g, ws in SHOWCASE.items() for _ in ws]
sub = counted.unit[[counted.vocab.index[w] for w in words]]
centered = sub - sub.mean(axis=0)
u, s, _ = np.linalg.svd(centered, full_matrices=False)
fig = viz.labeled_scatter(u[:, :2] * s[:2], words, groups,
    title='No labels and no training loop — words arrange themselves by meaning '
          'from co-occurrence counts alone')

## Path 2 — predict, and learn from mistakes

Skip-gram flips the logic: give each word a vector, use it to *predict* its neighbours, and
nudge the vectors whenever the prediction is wrong. With negative sampling the loss for a
(center $c$, context $o$) pair against $K$ sampled negatives is

$$-\log\sigma(u_o^\top v_c)\;-\;\sum_{k=1}^{K}\log\sigma(-u_{n_k}^\top v_c)$$

The three gradient lines in `embeddings/predict.py` are that formula differentiated by hand —
the same machinery Part 3 will build a full language model from.

In [ ]:
predicted = WordVectors(vocab, train_sgns(ids, vocab, seed=0, **SGNS_CFG))

for w in ('profit', 'rose'):
    print(f'{w:<7}', ', '.join(f'{n} {s:.2f}' for n, s in predicted.neighbors(w, 5)))

## The verdict

Twenty human-judged related pairs (*profit·earnings*, *staff·employees*, …) against 500 random
pairs: a good embedding should rank the related ones higher. AUC 0.5 is coin-flipping.

In [ ]:
models = {'PPMI + SVD (counting)': counted, 'Skip-gram (predicting)': predicted}
aucs = {k: pair_auc(m, RELATED) for k, m in models.items()}
for k, m in models.items():
    print(f'{k:<26} AUC {aucs[k]:.3f}   random-pair mean cosine {random_pair_mean(m):+.3f}')

fig = viz.bars(list(aucs), list(aucs.values()),
    title=f"Counting beats the neural network on this corpus — "
          f"AUC {max(aucs.values()):.2f} vs {min(aucs.values()):.2f}",
    xlabel='Related-vs-random pair AUC (0.5 = chance)', ref=0.5, ref_label='chance', fmt='{:.3f}')

Note the second column: the skip-gram model gives **random** word pairs a mean cosine near
+0.6 — it thinks everything resembles everything. That collapse of contrast (anisotropy) is a
known failure of gradient-trained embeddings on small corpora, and it is exactly what the AUC
punishes. 102k tokens is simply not enough data for the predictive path to shine;
Levy & Goldberg showed the two paths converge in what they *can* learn — but not in what they
need to learn it.

In [ ]:
floor = random_pair_mean(counted)
fig = viz.bars([f'{a} · {b}' for a, b in PROMISE],
    [counted.similarity(a, b) for a, b in PROMISE],
    title='The pairs Part 1 could not connect all sit far above the random floor',
    xlabel='Cosine similarity (PPMI + SVD)', ref=floor,
    ref_label=f'random pairs {floor:+.2f}', fmt='{:.2f}')

## Takeaways

1. **The signal was in the counts all along.** Part 1's model failed not because counting is
   weak, but because it never pooled evidence across contexts. PPMI + SVD is that pooling.
2. **Neural is not a synonym for better.** On 102k tokens, the no-training-loop method wins by
   a wide margin — data scale, not cleverness, is what feeds prediction-based learning.
3. **The gradients matter anyway.** Skip-gram loses the benchmark but supplies the machinery —
   embeddings trained by backprop — that every model from Part 3 onward is built on.

Run `python run.py` to regenerate every number and figure; `--sweep` reproduces the skip-gram
tuning table.